# WM-811K 量子自注意力 Transformer 公平比较实验

本 Notebook 使用 **PyTorch + DeepQuantum + Matplotlib**，比较四种共享同一晶圆图编码器和分类头的模型：

1. `quantum_transformer`：四量子比特参数化线路生成 Q/K/V，并完成输出投影和前馈变换；
2. `tiny_transformer`：经典多头自注意力；
3. `mlp_mixer`：纯 MLP 的全局 Token 交互；
4. `cnn_token_mixer`：深度可分离卷积的局部 Token 交互。

四个模型使用完全相同的数据、CNN Tokenizer、4 维 Token、分类头、优化器和训练协议，并通过参数量审计强制控制在量子模型的 ±5% 内。实验用于检验而不是预设量子模型一定领先；最终结论必须来自多随机种子统计。

DeepQuantum API 版本：4.5.0。参考：[顶层 API](https://dqapi.turingq.com/api/_autosummary/deepquantum.html)、[量子线路教程](https://dqapi.turingq.com/tutorials/basics.html)。

## 0. Anaconda 环境准备

建议在 Anaconda Prompt 中单独创建环境：

```bash
conda create -n qcs-wm811k python=3.11 -y
conda activate qcs-wm811k
# 按你的 CUDA 版本从 pytorch.org 选择 PyTorch 安装命令
pip install deepquantum==4.5.0 pandas numpy scikit-learn matplotlib tqdm psutil jupyter
python -m ipykernel install --user --name qcs-wm811k --display-name "Python (qcs-wm811k)"
```

如果当前内核已安装支持 GPU 的 PyTorch，只需运行下一单元。请勿为了方便而覆盖一个可用的 CUDA PyTorch。

In [ ]:
# 仅在缺少这些包时取消下一行注释并运行；PyTorch 请按本机 CUDA 版本单独安装。
# %pip install deepquantum==4.5.0 pandas numpy scikit-learn matplotlib tqdm psutil

from pathlib import Path
from dataclasses import replace
import json
import os
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
import torch
import matplotlib.pyplot as plt

from qcs_wm811k import (
    LABEL_NAMES, MODEL_NAMES, ExperimentConfig,
    environment_report, prepare_lswmd_cache, load_cache,
    class_distribution, make_lot_disjoint_split, split_class_distribution,
    parameter_audit,
    gradient_smoke_test, paired_seed_comparison,
    run_comparison_suite, run_few_shot_suite, evaluate_input_noise,
    plot_class_distribution, plot_training_histories, plot_result_bars,
    plot_noise_curves, plot_confusion_matrices,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
environment_report()

## 1. 路径与实验档位

原始 `LSWMD.pkl` 约 2.10 GB。旧版 pickle 无法真正流式读取，首次转换必须整体载入一次；建议关闭浏览器、游戏和其他占内存软件，并至少留出约 6–8 GB 可用内存。转换后只使用约百 MB 的轻量缓存。

默认采用快速自检档，它只用于验证代码是否能跑通，不能写入论文结论。

In [ ]:
PROJECT_DIR = Path.cwd()
# 默认读取项目内 data/LSWMD.pkl；也可通过 LSWMD_PATH 环境变量覆盖。
RAW_PKL = Path(os.environ.get('LSWMD_PATH', PROJECT_DIR / 'data' / 'LSWMD.pkl'))
CACHE_PATH = PROJECT_DIR / 'data_cache' / 'wm811k_labeled_32.npz'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'

# 第一次先使用快速档。全流程确认无误后改为 ExperimentConfig.publication()。
CONFIG = replace(ExperimentConfig.quick(), quantum_projection_mode='five')
# CONFIG = replace(ExperimentConfig.publication(), quantum_projection_mode='five')

print('Raw data:', RAW_PKL, RAW_PKL.exists())
print('Project:', PROJECT_DIR)
print(CONFIG)

## 2. 一次性生成轻量数据缓存

本步骤保留9个有标签类别，将不同尺寸晶圆图以最近邻方式转换成 32×32，避免在类别编码 0/1/2 之间制造无意义的小数。缓存同时保留 `lotName`，后续按制造批次做互斥划分。

In [ ]:
# 首次执行可能持续数分钟；如果缓存已经存在，则会立即返回。
prepare_lswmd_cache(
    raw_pickle=RAW_PKL,
    cache_path=CACHE_PATH,
    image_size=CONFIG.image_size,
    force=False,
)

In [ ]:
images, labels, lots = load_cache(CACHE_PATH)
display(class_distribution(labels))
fig = plot_class_distribution(labels)
plt.show()
print('images:', images.shape, images.dtype)
print('unique lots:', len(set(lots.tolist())))

## 3. 检查批次互斥划分与参数公平性

数据采用约70%/15%/15%的 train/validation/test 划分，并强制同一 `lotName` 不跨集合。参数审计超过±5%会直接报错，中止不公平实验。

In [ ]:
split = make_lot_disjoint_split(labels, lots, seed=2026)
split_summary = []
for split_name, idx in split.items():
    split_summary.append({
        'split': split_name,
        'samples': len(idx),
        'lots': len(set(lots[idx].tolist())),
        'classes_present': len(set(labels[idx].tolist())),
    })
display(pd.DataFrame(split_summary))
split_counts = split_class_distribution(labels, split)
display(split_counts.pivot(index='label', columns='split', values='count'))
display(parameter_audit(CONFIG, tolerance=0.05))
display(gradient_smoke_test(CONFIG))

## 4. 快速自检或完整公平比较

快速档：3轮、每类最多256个训练样本、单随机种子，只检查代码。

论文档：使用 Q/K/V、注意力输出和前馈分支共5个独立量子投影；建议60轮、每类最多2000个训练样本、5个随机种子。量子模拟即使使用GPU也可能耗时较长，建议先完成单种子实验，再运行五种子。测试集仅在验证集选好检查点后评价。

In [ ]:
# 快速自检：
SEEDS = (42,)

# 论文实验时改为：
# CONFIG = replace(ExperimentConfig.publication(), quantum_projection_mode='five')
# SEEDS = (42, 52, 62, 72, 82)

# 每次改变模型结构或配置都要使用新的目录，避免误读旧检查点。
RUN_DIR = ARTIFACT_DIR / 'five_quantum_projection_current_run'
results, histories, confusion_matrices = run_comparison_suite(
    cache_path=CACHE_PATH,
    config=CONFIG,
    seeds=SEEDS,
    split_seed=2026,
    artifact_dir=RUN_DIR,
)
display(results)

In [ ]:
summary = results.groupby('model').agg(
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    balanced_acc_mean=('balanced_accuracy', 'mean'),
    balanced_acc_std=('balanced_accuracy', 'std'),
    parameters=('parameters', 'first'),
    train_seconds_mean=('train_seconds', 'mean'),
).sort_values('macro_f1_mean', ascending=False)
display(summary)
display(paired_seed_comparison(results, metric='macro_f1'))

plot_training_histories(histories, seed=SEEDS[0]); plt.show()
plot_result_bars(results, metric='macro_f1'); plt.show()
plot_result_bars(results, metric='balanced_accuracy'); plt.show()
plot_confusion_matrices(confusion_matrices, seed=SEEDS[0]); plt.show()

## 5. 预先固定的次要假设与判定方法

主实验完成后再检验两个次要假设，不得根据测试集结果重新调整模型或超参数：

1. **H1 少样本假设**：在每类最多 25、50、100、200 个训练样本时，量子 Transformer 的平均 Macro-F1 高于参数量匹配的经典基线。主判定量是每个种子在四个 K 值上的 Macro-F1 平均值；各 K 结果作支持性分析。
2. **H2 噪声鲁棒性假设**：在 10% die 状态扰动下，量子 Transformer 相对无噪声性能的 Macro-F1 保留率高于经典基线。1%、3%、5% 作支持性曲线分析。

两项实验均使用预先固定的 5 个种子 `(42, 52, 62, 72, 82)` 和主实验划分种子 `20260815`。同时报告均值、标准差、配对差、95% 置信区间和胜率。

## 6. H1：少样本学习实验

每个 K 和种子都需要重新训练四个模型，验证集和测试集保持完整。为减少 Kernel 崩溃风险，下面一次只运行一个 `(K, seed)` 任务。

In [ ]:
FINAL_SPLIT_SEED = 20260815
SECONDARY_SEEDS = (42, 52, 62, 72, 82)
K_VALUES = (25, 50, 100, 200)
FEW_SHOT_ROOT = ARTIFACT_DIR / 'few_shot_publication'

FEW_SHOT_BASE_CONFIG = replace(
    ExperimentConfig.publication(),
    epochs=60,
    patience=12,
    batch_size=64,
    sampler_power=0.5,
    quantum_init_scale=0.1,
    quantum_projection_mode='five',
    eval_cap_per_class=None,
)

print('Few-shot output:', FEW_SHOT_ROOT)
print('K values:', K_VALUES, 'Seeds:', SECONDARY_SEEDS)

In [ ]:
# 每次只修改这两个值。完成后依次运行其他组合。
FEW_K = 25
FEW_SEED = 42

assert FEW_K in K_VALUES
assert FEW_SEED in SECONDARY_SEEDS
few_config = replace(FEW_SHOT_BASE_CONFIG, train_cap_per_class=FEW_K)
few_dir = FEW_SHOT_ROOT / f'k_{FEW_K}' / f'seed_{FEW_SEED}'
result_file = few_dir / 'comparison_results.csv'

if result_file.exists():
    old_result = pd.read_csv(result_file)
else:
    old_result = pd.DataFrame()

if set(old_result.get('model', [])) == set(MODEL_NAMES):
    print('This task is already complete; existing result was loaded:', result_file)
    few_results = old_result
else:
    if len(old_result):
        print('Warning: a partial result exists and this run will retrain this task.')
    few_results, few_histories, few_matrices = run_comparison_suite(
        cache_path=CACHE_PATH,
        config=few_config,
        seeds=(FEW_SEED,),
        split_seed=FINAL_SPLIT_SEED,
        artifact_dir=few_dir,
    )

display(few_results)

完成 20 个 `(K, seed)` 组合后运行下面的汇总单元。若任务未全部完成，它只会列出缺少项，不会将不完整结果写入总表。

In [ ]:
few_frames = []
missing_few_tasks = []

for k in K_VALUES:
    for seed in SECONDARY_SEEDS:
        path = FEW_SHOT_ROOT / f'k_{k}' / f'seed_{seed}' / 'comparison_results.csv'
        if not path.exists():
            missing_few_tasks.append((k, seed, 'file missing'))
            continue
        frame = pd.read_csv(path)
        if set(frame['model']) != set(MODEL_NAMES):
            missing_few_tasks.append((k, seed, 'models incomplete'))
            continue
        frame.insert(0, 'shots_per_class', k)
        few_frames.append(frame)

if missing_few_tasks:
    print('Unfinished few-shot tasks:', missing_few_tasks)
else:
    few_shot_results = pd.concat(few_frames, ignore_index=True)
    FEW_SHOT_ROOT.mkdir(parents=True, exist_ok=True)
    few_shot_results.to_csv(FEW_SHOT_ROOT / 'few_shot_results_5seeds.csv', index=False)

    few_summary = (few_shot_results.groupby(['shots_per_class', 'model'])
        .agg(macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
             balanced_acc_mean=('balanced_accuracy', 'mean'),
             balanced_acc_std=('balanced_accuracy', 'std'))
        .reset_index())
    display(few_summary)

    # H1 主判定量：各种子在四个 K 值上的平均 Macro-F1。
    h1_primary = (few_shot_results.groupby(['seed', 'model'], as_index=False)
        .agg(macro_f1=('macro_f1', 'mean')))
    display(paired_seed_comparison(h1_primary, metric='macro_f1'))

In [ ]:
if not missing_few_tasks:
    fig, ax = plt.subplots(figsize=(8, 5))
    for model, group in few_summary.groupby('model'):
        group = group.sort_values('shots_per_class')
        ax.errorbar(group['shots_per_class'], group['macro_f1_mean'],
                    yerr=group['macro_f1_std'], marker='o', capsize=4, label=model)
    ax.set_xlabel('Maximum training samples per class')
    ax.set_ylabel('Test Macro-F1')
    ax.set_title('Few-shot performance on WM-811K')
    ax.set_xticks(K_VALUES)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FEW_SHOT_ROOT / 'few_shot_macro_f1.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. H2：输入噪声鲁棒性实验

本实验不重新训练，而是读取主实验保存的检查点，对固定测试集的有效 die 状态加入 0%、1%、3%、5%、10% 的确定性扰动。它表示制造测量输入噪声，不是量子门噪声。四个模型在同一种子下看到完全相同的扰动。

请将 `FINAL_RUN_ROOT` 改为五个主实验种子目录的共同父目录，其下应有 `seed_42`、`seed_52`、`seed_62`、`seed_72`、`seed_82`。

In [ ]:
NOISE_LEVELS = (0.00, 0.01, 0.03, 0.05, 0.10)
FINAL_RUN_ROOT = ARTIFACT_DIR / 'publication_final'  # 如有不同，只修改这一行

for seed in SECONDARY_SEEDS:
    seed_dir = FINAL_RUN_ROOT / f'seed_{seed}'
    print(seed, 'directory:', seed_dir.exists(),
          'config:', (seed_dir / 'config.json').exists(),
          'checkpoints:', (seed_dir / 'checkpoints').exists())

In [ ]:
# 每次只修改这一个种子。从 config.json 恢复训练配置，防止检查点不匹配。
NOISE_SEED = 42
assert NOISE_SEED in SECONDARY_SEEDS
seed_dir = FINAL_RUN_ROOT / f'seed_{NOISE_SEED}'
config_file = seed_dir / 'config.json'

if not config_file.exists():
    raise FileNotFoundError(f'Please correct FINAL_RUN_ROOT; missing: {config_file}')

with config_file.open('r', encoding='utf-8') as handle:
    noise_config = ExperimentConfig(**json.load(handle))

noise_results_one_seed = evaluate_input_noise(
    cache_path=CACHE_PATH,
    config=noise_config,
    seed=NOISE_SEED,
    noise_levels=NOISE_LEVELS,
    artifact_dir=seed_dir,
)
display(noise_results_one_seed[['model', 'noise', 'macro_f1', 'balanced_accuracy']])

五个种子全部完成后汇总。H2 主指标为 10% 噪声下 `Macro-F1 retention = noisy Macro-F1 / clean Macro-F1`，越高越好。性能下降值和完整噪声曲线作支持性结果。

In [ ]:
noise_frames = []
missing_noise_seeds = []

for seed in SECONDARY_SEEDS:
    path = FINAL_RUN_ROOT / f'seed_{seed}' / 'input_noise_results.csv'
    if not path.exists():
        missing_noise_seeds.append(seed)
        continue
    frame = pd.read_csv(path)
    expected_rows = len(MODEL_NAMES) * len(NOISE_LEVELS)
    if len(frame) != expected_rows or set(frame['model']) != set(MODEL_NAMES):
        missing_noise_seeds.append(seed)
        continue
    frame.insert(1, 'seed', seed)
    noise_frames.append(frame)

if missing_noise_seeds:
    print('Unfinished noise seeds:', missing_noise_seeds)
else:
    noise_results = pd.concat(noise_frames, ignore_index=True)
    noise_results.to_csv(FINAL_RUN_ROOT / 'input_noise_results_5seeds.csv', index=False)

    clean = (noise_results[noise_results['noise'] == 0]
             [['seed', 'model', 'macro_f1']]
             .rename(columns={'macro_f1': 'clean_macro_f1'}))
    noise_analysis = noise_results.merge(clean, on=['seed', 'model'], how='left')
    noise_analysis['macro_f1_drop'] = noise_analysis['clean_macro_f1'] - noise_analysis['macro_f1']
    noise_analysis['macro_f1_retention'] = (
        noise_analysis['macro_f1'] / noise_analysis['clean_macro_f1'].clip(lower=1e-12))

    noise_summary = (noise_analysis.groupby(['noise', 'model'])
        .agg(macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
             retention_mean=('macro_f1_retention', 'mean'),
             retention_std=('macro_f1_retention', 'std'),
             drop_mean=('macro_f1_drop', 'mean'), drop_std=('macro_f1_drop', 'std'))
        .reset_index())
    display(noise_summary)

    # H2 主检验：10% 噪声下的 Macro-F1 保留率。
    h2_primary = noise_analysis[noise_analysis['noise'] == 0.10].copy()
    display(paired_seed_comparison(h2_primary, metric='macro_f1_retention'))

In [ ]:
if not missing_noise_seeds:
    fig, ax = plt.subplots(figsize=(8, 5))
    for model, group in noise_summary.groupby('model'):
        group = group.sort_values('noise')
        ax.errorbar(group['noise'] * 100, group['macro_f1_mean'],
                    yerr=group['macro_f1_std'], marker='o', capsize=4, label=model)
    ax.set_xlabel('Corrupted die states (%)')
    ax.set_ylabel('Test Macro-F1')
    ax.set_title('Input-noise robustness on WM-811K')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FINAL_RUN_ROOT / 'noise_robustness_macro_f1.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. 论文结果判定规则

- H1 主检验使用四个 K 值上的平均 Macro-F1；H2 主检验使用 10% 噪声下的 Macro-F1 保留率。
- 五个种子是配对统计单位；不能把每张晶圆当成独立统计重复。
- 只有当量子模型对某个基线的配对差为正，且 95% 置信区间支持该差异时，才能对该基线声称统计优势。
- 若均值领先但置信区间跨过 0，应写成‘呈现正向趋势，但未达到统计显著’；若未稳定领先，应写成‘未观察到稳定优势’。
- 结果来自 DeepQuantum 经典 GPU 模拟，不代表真实量子硬件上的速度、功耗或噪声优势。

次要实验输出保存在 `artifacts/few_shot_publication/` 和主实验各 `seed_*` 目录中。